[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ItunuAjiboye/Tutorial_Host_Pathogen_Protein_Protein_Interaction_Prediction/blob/main/notebooks/03_Feature_Extraction.ipynb)
> This notebook is designed to be run in Google Colab. Click on "Open in Colab" to continue.

# **Deployment: Final Model Refit and Prediction on New HP-PPI Pairs**

## Overview

This notebook takes the best-performing model selected and evaluated in
Notebook 5 (Hyperparameter Tuning & LOPO Validation) and prepares it for
deployment: refitting on the complete dataset, and applying it to new,
previously unlabeled host-pathogen protein pairs.

    IMPORTANT: This notebook does NOT recompute or replace any reported
    hold-out or LOPO evaluation metrics from Notebook 5. Those metrics
    come from a model trained on the training partition only, before
    ever seeing the hold-out test set that remains the valid basis for
    all reported performance numbers. This notebook produces a SEPARATE
    model,  refit on the full dataset, used only for prediction on
    genuinely new pairs.

This notebook is organized into two sections:
1. **Deployment Model Refit**: Refit the selected best model (from
   Notebook 5) on the combined training and hold-out data, and save it.


2. **Prediction on New HP-PPI Pairs**: Load the saved deployment model
   and apply it to new, feature-extracted host-pathogen pairs.


In [1]:
#Mounting Google Drive to access files
import os
from google.colab import drive
drive.mount ('/content/my_drive')

#import pandas for data manipulation
import pandas as pd


Mounted at /content/my_drive


##Section 0: Select ratio and feature set

In [8]:
# SECTION 0: SELECT RATIO AND FEATURE SET
# =============================================================================
# Must match the ratio/feature combination of the model you want to deploy.

CURRENT_RATIO = 10   # <-- change this: 1, 5, or 10
feature_name = "AAC" # <--- change to desired feature set

base_hpi_path = "/content/my_drive/MyDrive/HPI"

ratio_config = {
    1:  {"features_dir": f"{base_hpi_path}/Features/Balanced", "ml_dir": f"{base_hpi_path}/ML/Balanced", "label": "1:1"},
    5:  {"features_dir": f"{base_hpi_path}/Features/R5",       "ml_dir": f"{base_hpi_path}/ML/R5",       "label": "1:5"},
    10: {"features_dir": f"{base_hpi_path}/Features/R10",      "ml_dir": f"{base_hpi_path}/ML/R10",      "label": "1:10"},
}

cfg = ratio_config[CURRENT_RATIO]
features_dir = cfg["features_dir"]
ml_dir = cfg["ml_dir"]
label_tag = cfg["label"].replace(":", "_")

print(f"Ratio: {cfg['label']} | Features: {feature_name} | ml_dir: {ml_dir}")


Ratio: 1:10 | Features: AAC | ml_dir: /content/my_drive/MyDrive/HPI/ML/R10


##Section 1: Function definitions (run once per session)

In [9]:
# SECTION 1: FUNCTION DEFINITIONS
# =============================================================================

import joblib
from sklearn.base import clone

# Must match non_feature_cols used throughout Notebooks 2-5
non_feature_cols = [
    "host_sequence", "pathogen_sequence",
    "host_protein_id", "pathogen_protein_id",
    "host_cluster", "pathogen_cluster",
    "pathogen_specie", "host_partition",
    "pathogen_partition", "label",
    "_fold_id", "_cv_fold",
]

def get_X_y(df):
    feature_cols = [c for c in df.columns if c not in non_feature_cols]
    X = df[feature_cols].astype(float)
    y = df["label"].astype(int)
    return X, y

def predict_new_pairs(new_df, model, id_cols=("host_protein_id", "pathogen_protein_id")):
    """
    Predicts interaction status for new host-pathogen pairs.

    new_df must already be preprocessed and feature-extracted using the
    same descriptor (feature_name) the deployment model was trained on.
    A dummy "label" column is added only because get_X_y() expects one —
    it is dropped immediately and never used in prediction.
    """
    X_new, _ = get_X_y(new_df.assign(label=0))

    y_pred = model.predict(X_new)
    y_prob = model.predict_proba(X_new)[:, 1]

    out_cols = [c for c in id_cols if c in new_df.columns]
    result = new_df[out_cols].copy() if out_cols else pd.DataFrame(index=new_df.index)
    result["predicted_label"] = y_pred            # 1 = predicted interaction, 0 = no interaction
    result["predicted_probability"] = y_prob       # model confidence, 0-1
    return result

print("Section 1 complete: all functions defined.")

Section 1 complete: all functions defined.


##Section 2: Deployment Model (Refit on Full Data)

This section refits the selected best model on the complete dataset
(train_df + test_df combined) to produce a deployment-ready model for
predicting new, unlabeled host-pathogen pairs.

    This section requires that Notebook 5 has already been run at least
    once for this ratio/feature combination, so that the best-model
    selection file and tuned model checkpoint exist on disk.


####Step 1: Load the tuned best model and data

In [10]:
# SECTION 2, STEP 1: LOAD TUNED BEST MODEL AND DATA
# =============================================================================

best_model_selection_df = pd.read_csv(f"{ml_dir}/{label_tag}_{feature_name}_best_model_selection.csv")
best_model_name = best_model_selection_df["Best_Model"].iloc[0]

tuned_model_path = f"{ml_dir}/{label_tag}_{feature_name}_tuned_{best_model_name}.pkl"
best_model = joblib.load(tuned_model_path)

train_df = pd.read_csv(f"{features_dir}/train_{feature_name}_split_features.csv")
test_df = pd.read_csv(f"{features_dir}/test_{feature_name}_split_features.csv")

print(f"Ratio: {cfg['label']} | Feature: {feature_name} | Best model: {best_model_name}")


Ratio: 1:10 | Feature: AAC | Best model: LightGBM


####Step 2: Refit on full data and save as the deployment model

In [11]:
# SECTION 2, STEP 2: REFIT ON FULL DATA, SAVE DEPLOYMENT MODEL
# =============================================================================

full_df = pd.concat([train_df, test_df], ignore_index=True)
X_full, y_full = get_X_y(full_df)

deployment_model = clone(best_model)   # fresh, unfitted clone with the same tuned hyperparameters
deployment_model.fit(X_full, y_full)

deployment_model_path = f"{ml_dir}/{label_tag}_{feature_name}_deployment_model_{best_model_name}.pkl"
joblib.dump(deployment_model, deployment_model_path)

print(f"Deployment model refit on {len(full_df)} pairs (train+hold-out combined) "
      f"and saved to {deployment_model_path}")



Deployment model refit on 48917 pairs (train+hold-out combined) and saved to /content/my_drive/MyDrive/HPI/ML/R10/1_10_AAC_deployment_model_LightGBM.pkl


##Section 3: Predicting New HP-PPI Pairs

This section applies the saved deployment model (Section 2 above) to a new
set of host-pathogen protein pairs. It is fully resumable independently,
if your session restarted, you can jump straight here after rerunning
Section 0 and Section 1, without repeating the deployment refit, as long
as the deployment model has already been saved once.

    Prerequisite: The new pairs must already be preprocessed and feature-
    extracted using the SAME feature representation (feature_name) the
    deployment model was trained on. No homology clustering is needed for
    new pairs — clustering is only used when constructing train/CV/
    hold-out splits, not at inference time.

####Step 1: Load the saved deployment model


In [ ]:
# SECTION 3, STEP 1: LOAD SAVED DEPLOYMENT MODEL
# =============================================================================

best_model_selection_df = pd.read_csv(f"{ml_dir}/{label_tag}_{feature_name}_best_model_selection.csv")
best_model_name = best_model_selection_df["Best_Model"].iloc[0]
deployment_model_path = f"{ml_dir}/{label_tag}_{feature_name}_deployment_model_{best_model_name}.pkl"

print(f"Ratio: {cfg['label']} | Feature: {feature_name} | Best model: {best_model_name}")
print(f"Loading deployment model from: {deployment_model_path}")

deployment_model = joblib.load(deployment_model_path)

####Step 2: Predict on new pairs

In [ ]:
# SECTION 3, STEP 2: PREDICT NEW PAIRS
# =============================================================================

# new_df = pd.read_csv(f"{features_dir}/new_pairs_{feature_name}_features.csv")
# new_predictions_df = predict_new_pairs(new_df, deployment_model)
# new_predictions_df.to_csv(f"{ml_dir}/{label_tag}_{feature_name}_new_ppi_predictions.csv", index=False)
# new_predictions_df.head()